In [1]:
from pathlib import Path
from datetime import datetime, timezone
import re, json, math
import pandas as pd
from collections import Counter

RAW_DIR = Path("/home/ebb/llm-hackathon/discord/raw/issues")   # the raw JSON threads by joe
OUT_DIR = Path("/home/ebb/llm-hackathon/data/discord")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def slugify(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"[^a-z0-9]+", "-", s)
    s = re.sub(r"-+", "-", s).strip("-")
    return s or "untitled"

def to_utc_iso(ts: str) -> str:
    if not ts:
        return ""
    try:
        dt = datetime.fromisoformat(ts)
        return dt.astimezone(timezone.utc).replace(tzinfo=timezone.utc).isoformat().replace("+00:00", "Z")
    except Exception:
        return ts

json_files = sorted(RAW_DIR.glob("*.json"))
len(json_files), [p.name for p in json_files[:5]]


(303,
 ['NOMAD - issues - "This entry does not exist." upon publishing [1275052785494392914].json',
  'NOMAD - issues - # not escaped in API calls [1301134820063313992].json',
  'NOMAD - issues - .hdf5 files (own made .h5, .h5iona, ...) for publication on central nomad [1374638026356949032].json',
  'NOMAD - issues - 1.3.14? [1338856302100873267].json',
  'NOMAD - issues - 2D array of strings [1250003438503333938].json'])

In [2]:
def normalize_name(name: str) -> str:
    s = (name or "").lstrip("@").strip()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[\u200b-\u200f\u202a-\u202e]", "", s)
    return s

def rewrite_mentions(text: str, mentions: list) -> str:
    if not text or not mentions:
        return text or ""
    reps = []
    for m in mentions:
        nick = m.get("nickname") or m.get("name")
        name = m.get("name")
        cands = []
        if nick: cands.append(nick)
        if name and name != nick: cands.append(name)
        uniq = sorted(set(cands), key=lambda s: len(s), reverse=True)
        norm = normalize_name(nick or name or "")
        if not norm:
            continue
        for cand in uniq:
            reps.append((r"@" + re.escape(cand), f"@{norm}"))
    seen, ordered = set(), []
    for pat, repl in reps:
        if pat not in seen:
            seen.add(pat); ordered.append((pat, repl))
    out = text
    for pat, repl in ordered:
        out = re.sub(pat, repl, out)
    return out

def redact_pii(text: str) -> str:
    # emails
    text = re.sub(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}", "[REDACTED_EMAIL]", text)
    # rough phones
    text = re.sub(r"\b(?:\+?\d{1,3}[-.\s]?)?(?:\(?\d{2,4}\)?[-.\s]?){2,4}\d{2,4}\b", "[REDACTED_PHONE]", text)
    return text

def load_thread_as_text(path: Path):
    data = json.loads(path.read_text(encoding="utf-8"))
    guild = data.get("guild", {}) or {}
    channel = data.get("channel", {}) or {}
    msgs = data.get("messages", []) or []

    title   = channel.get("name") or path.stem
    section = channel.get("category") or guild.get("name")
    first_ts = None

    # concatenate messages into one textual stream per thread (keeps order)
    texts = []
    for m in msgs:
        t = (m.get("content") or "").strip()
        if not t:
            continue
        t = rewrite_mentions(t, m.get("mentions") or [])
        t = redact_pii(t)
        if len(t) < 3:
            continue
        # author = (m.get("author", {}) or {}).get("nickname") or (m.get("author", {}) or {}).get("name")
        # t = f"{author}: {t}"
        texts.append(t)
        if not first_ts:
            first_ts = to_utc_iso(m.get("timestamp"))
    full_text = "\n".join(texts)
    return {
        "source": "discord",
        "title": title,
        "section": section,
        "text": full_text,
        "url": None,
        "timestamp": first_ts or ""
    }

threads = [load_thread_as_text(p) for p in json_files]
len(threads), threads[0]["title"], len(threads[0]["text"])


(303, '"This entry does not exist." upon publishing', 3484)

In [3]:
def tokenize_words(text: str):
    return re.findall(r"[A-Za-zÄÖÜäöüß0-9]+", text or "")

def split_sentences(text: str):
    parts = re.split(r"(?<=[.!?])\s+(?=[A-ZÄÖÜ])", (text or "").strip())
    return [p.strip() for p in parts if p.strip()] or [text.strip()]

def chunk_fixed(text: str, max_words: int = 400, overlap: int = 50):
    toks = tokenize_words(text)
    if not toks:
        return []
    chunks, i, N = [], 0, len(toks)
    while i < N:
        j = min(i + max_words, N)
        chunks.append(" ".join(toks[i:j]))
        if j == N:
            break
        i = max(0, j - overlap)
    return chunks

def chunk_sentence(text: str, target_words: int = 350):
    sents = split_sentences(text)
    chunks, buf, count = [], [], 0
    for s in sents:
        n = len(tokenize_words(s))
        if count + n > target_words and buf:
            chunks.append(" ".join(buf)); buf, count = [], 0
        buf.append(s); count += n
    if buf:
        chunks.append(" ".join(buf))
    return chunks

def bow_vector(text: str) -> Counter:
    return Counter(tokenize_words((text or "").lower()))

def cosine_sim(a: Counter, b: Counter) -> float:
    if not a or not b:
        return 0.0
    inter = set(a) & set(b)
    num = sum(a[t]*b[t] for t in inter)
    da = math.sqrt(sum(v*v for v in a.values()))
    db = math.sqrt(sum(v*v for v in b.values()))
    if da == 0 or db == 0:
        return 0.0
    return num / (da * db)

def chunk_semantic(text: str, target_words: int = 350, min_sim: float = 0.25):
    sents = split_sentences(text)
    if not sents:
        return []
    chunks, buf, centroid, count = [], [], Counter(), 0
    for s in sents:
        sv = bow_vector(s)
        sim = cosine_sim(centroid, sv) if centroid else 1.0
        n = len(tokenize_words(s))
        if (sim < min_sim and buf) or (count + n > target_words and buf):
            chunks.append(" ".join(buf)); buf, centroid, count = [], Counter(), 0
        buf.append(s); centroid.update(sv); count += n
    if buf:
        chunks.append(" ".join(buf))
    return chunks


In [4]:
FIX_MAX, FIX_OVERLAP   = 400, 50
SENT_TARGET            = 350
SEM_TARGET, SEM_MINSIM = 350, 0.25

out_fixed    = OUT_DIR / "nomad_discord.fixed.jsonl"
out_sentence = OUT_DIR / "nomad_discord.sentence.jsonl"
out_semantic = OUT_DIR / "nomad_discord.semantic.jsonl"

def write_jsonl(rows, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

fixed_rows, sentence_rows, semantic_rows = [], [], []

for thr in threads:
    title   = thr["title"]
    section = thr["section"]
    url     = thr["url"]
    ts      = thr["timestamp"]
    text    = thr["text"]

    # fixed
    for i, ch in enumerate(chunk_fixed(text, max_words=FIX_MAX, overlap=FIX_OVERLAP)):
        fixed_rows.append({
            "id": f"discord:{slugify(title)}:{slugify(section)}:fixed:{i}",
            "source": "discord",
            "title": title,
            "section": section,
            "text": ch,
            "url": url,
            "timestamp": ts
        })
    # sentence
    for i, ch in enumerate(chunk_sentence(text, target_words=SENT_TARGET)):
        sentence_rows.append({
            "id": f"discord:{slugify(title)}:{slugify(section)}:sentence:{i}",
            "source": "discord",
            "title": title,
            "section": section,
            "text": ch,
            "url": url,
            "timestamp": ts
        })
    # semantic
    for i, ch in enumerate(chunk_semantic(text, target_words=SEM_TARGET, min_sim=SEM_MINSIM)):
        semantic_rows.append({
            "id": f"discord:{slugify(title)}:{slugify(section)}:semantic:{i}",
            "source": "discord",
            "title": title,
            "section": section,
            "text": ch,
            "url": url,
            "timestamp": ts
        })

write_jsonl(fixed_rows, out_fixed)
write_jsonl(sentence_rows, out_sentence)
write_jsonl(semantic_rows, out_semantic)

len(fixed_rows), len(sentence_rows), len(semantic_rows), out_fixed.as_posix(), out_sentence.as_posix(), out_semantic.as_posix()


(545,
 599,
 3418,
 '/home/ebb/llm-hackathon/data/discord/nomad_discord.fixed.jsonl',
 '/home/ebb/llm-hackathon/data/discord/nomad_discord.sentence.jsonl',
 '/home/ebb/llm-hackathon/data/discord/nomad_discord.semantic.jsonl')

In [5]:
# size & a peek
for p in [out_fixed, out_sentence, out_semantic]:
    n = sum(1 for _ in p.open("r", encoding="utf-8"))
    print(p.name, "lines:", n)

# load a few rows
sample = []
with out_sentence.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5: break
        sample.append(json.loads(line))
pd.DataFrame([{
    "id": r["id"], "title": r["title"], "section": r["section"], "chars": len(r["text"]), "preview": r["text"][:200]
} for r in sample])


nomad_discord.fixed.jsonl lines: 545
nomad_discord.sentence.jsonl lines: 599
nomad_discord.semantic.jsonl lines: 3418


,id,title,section,chars,preview
0,discord:this-entry-does-not-exist-upon-publish...,"""This entry does not exist."" upon publishing",issues,1861,"Hi everyone, here's an issue I'm seeing when p..."
1,discord:this-entry-does-not-exist-upon-publish...,"""This entry does not exist."" upon publishing",issues,1621,"If you encounter this for other of your files,..."
2,discord:not-escaped-in-api-calls:issues:senten...,# not escaped in API calls,issues,436,"Hey all,\nI have file names including # and i ..."
3,discord:hdf5-files-own-made-h5-h5iona-for-publ...,".hdf5 files (own made .h5, .h5iona, ...) for p...",issues,1602,I have to upload all sorts of .hdf5 files (own...
4,discord:hdf5-files-own-made-h5-h5iona-for-publ...,".hdf5 files (own made .h5, .h5iona, ...) for p...",issues,2472,Is there any doc about that?\nin order to make...
